In [0]:
from pyspark.sql.functions import col# Manual simulation of what DLT does automatically:

# 1. Bronze: read new files
# 2. Silver: apply "expectations" as filters, tracking metrics
# 3. Gold: aggregate

CATALOG = "insurance_dev"

# BRONZE (batch read simulating the streaming source)
df_bronze_sim = spark.read.json(f"/Volumes/{CATALOG}/raw_files/claims_stream/")
df_bronze_sim.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.bronze.claims_stream_sim")

# SILVER with manual "expectation" tracinsurance_dev.bronze.claims_stream_simking (mimics DLT's constraint metrics)
total_rows = df_bronze_sim.count()
passed = df_bronze_sim.filter((col("claim_amount") > 0) & col("claim_id").isNotNull())
dropped = total_rows - passed.count()

print(f"📊 Simulated DLT Expectation Metrics:")
print(f"   Total rows in:  {total_rows}")
print(f"   Passed:         {passed.count()}")
print(f"   Dropped:        {dropped}  ({round(dropped/total_rows*100,1)}%)")

passed.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.claims_stream_sim_clean")

# GOLD
spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG}.gold.claims_stream_summary AS
    SELECT claim_status, COUNT(*) claim_count, SUM(claim_amount) total_amount
    FROM {CATALOG}.silver.claims_stream_sim_clean
    GROUP BY claim_status
""")

display(spark.table(f"{CATALOG}.gold.claims_stream_summary"))

📊 Simulated DLT Expectation Metrics:
   Total rows in:  21
   Passed:         14
   Dropped:        7  (33.3%)


claim_status,claim_count,total_amount
UnderReview,6,26000.0
Approved,1,3000.0
Rejected,2,17000.0
Filed,5,35000.0
